# 📥 PASO 0: DESCARGAR DANY AI STUDIO A TU ORDENADOR (PC WINDOWS)
### ¿Quieres instalar Dany AI Studio en tu PC? ¡Ejecuta la celda de abajo!

> **¿Qué es Dany AI Studio?**  
> Es una suite completa de producción musical y audiovisual con IA que corre **en tu propio ordenador con Windows** (a través de `start_studio.bat`).  
> Este cuaderno de Google Colab funciona como **servidor acelerador en la nube** para cuando quieras delegar renders o música pesada a una GPU en la nube.

**Instrucciones rápidas para instalar en tu PC:**
1. **Ejecuta la celda de código inferior** para descargar `Dany_AI_Studio_v2.7_Gratis.zip` a tu navegador.
2. **Descomprime el ZIP** en cualquier carpeta de tu PC (por ejemplo `C:\Dany_AI_Studio` o en tu carpeta de Música).
3. **Ejecuta `install.bat`** (solo una vez) para verificar Python y librerías.
4. **Haz doble clic en `start_studio.bat`**: ¡se abrirá la interfaz del Studio en tu navegador web!


In [ ]:
# 📥 CELDA DE DESCARGA DIRECTA DE DANY AI STUDIO (ZIP)
import os, shutil
from google.colab import files

print('🚀 Buscando instalador de Dany AI Studio...')

ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Produccion_Musical/Dany_AI_Studio_v2.7_Gratis.zip',
    '/content/drive/MyDrive/Produccion_Musical/Dany_AI_Studio_v2.6_Gratis.zip',
    '/content/drive/MyDrive/Produccion_Musical/Dany_AI_Studio_v2.0_Gratis.zip',
    '/content/Dany_AI_Studio_v2.7_Gratis.zip',
]

found_zip = next((z for z in ZIP_CANDIDATES if os.path.exists(z)), None)

if found_zip:
    print(f'✅ Instalador encontrado: {os.path.basename(found_zip)}')
    print('⏳ Iniciando descarga automática a tu PC...')
    files.download(found_zip)
else:
    studio_dir = '/content/drive/MyDrive/Produccion_Musical/Dany_AI_Studio'
    if os.path.exists(studio_dir):
        out_zip = '/content/Dany_AI_Studio_v2.7_Gratis.zip'
        print('📦 Generando paquete ZIP desde Google Drive...')
        shutil.make_archive('/content/Dany_AI_Studio_v2.7_Gratis', 'zip', studio_dir)
        print('⏳ Iniciando descarga automática a tu PC...')
        files.download(out_zip)
    else:
        print('ℹ️ Si eres el autor o tienes acceso a la carpeta de Drive, ejecuta primero el PASO 1 (Montar Google Drive) y vuelve a ejecutar esta celda.')
        print('ℹ️ Si eres seguidor, también puedes descargar el ZIP desde el enlace compartido de Google Drive en la descripción de YouTube.')


# 🚀 DANY AI STUDIO - SERVIDOR COLAB DRIVE BRIDGE (v2.1 seguro)
### Producción Musical y Vídeo en la Nube (Gratis en Google Colab)

Este cuaderno es el **trabajador en la nube** para música pesada (ACE-Step 1.5 XL), renders (LTX-2.5 / ComfyUI) y entrenamientos LoRA.

**Flujo:**
1. Tu PC deja un trabajo **firmado (HMAC-SHA256)** en `Colab_Bridge/jobs/`.
2. Este cuaderno **verifica la firma**, la caducidad y que el script esté en la allowlist, y solo entonces lo ejecuta (sin shell).
3. Resultados y logs vuelven a tu Drive.

**Por qué es seguro:** ya no se ejecuta cualquier `.json` que aparezca en la carpeta (antes, cualquiera con acceso de escritura a esa carpeta de Drive controlaba esta máquina, que además tiene tu Drive montado).

**Configuración única (1 minuto):**
1. En tu PC: `python studio_core.py --show-colab-secret`
2. En Colab: icono 🔑 **Secrets** → nuevo secreto `DANY_COLAB_HMAC` → pega el valor → activa *Acceso desde el cuaderno*.
3. Ejecuta las celdas en orden.

In [ ]:
# PASO 1: Montar Google Drive y verificar acelerador GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# PASO 2: Carpetas del Colab Bridge en Google Drive
import os

BASE_BRIDGE = '/content/drive/MyDrive/Produccion_Musical/Colab_Bridge'
SUBDIRS = ('jobs', 'results', 'logs', 'outputs', 'scripts', 'processed', 'rejected', 'failed', 'state')
for sub in SUBDIRS:
    os.makedirs(f'{BASE_BRIDGE}/{sub}', exist_ok=True)

print('✅ Carpetas del Colab Bridge listas:')
for sub in SUBDIRS:
    print(f'   - {BASE_BRIDGE}/{sub}')

In [ ]:
# PASO 3: Seguridad - secreto compartido y política de ejecución
import os, re, json, time, hmac, hashlib, shlex, shutil, subprocess, threading, glob
from google.colab import userdata

# Secreto HMAC (mismo valor que en tu PC: python studio_core.py --show-colab-secret)
try:
    SECRET = userdata.get('DANY_COLAB_HMAC')
except Exception as exc:
    raise SystemExit(f'Falta el secreto DANY_COLAB_HMAC en Colab Secrets ({exc}). Sin él, el puente NO ejecuta nada.')
if not SECRET or len(SECRET) < 20:
    raise SystemExit('DANY_COLAB_HMAC está vacío o es demasiado corto.')

# Política. Debe ser igual o MÁS estricta que config.json -> colab de tu PC.
MAX_AGE_S = 600                 # un trabajo caduca a los 10 minutos
MAX_JOB_SECONDS = 4 * 3600      # tiempo máximo de ejecución por trabajo
ALLOWED_EXECUTABLES = {'python', 'python3'}
ALLOWED_SCRIPTS = [
    '/content/*.py',
    '/content/ACE-Step-1.5/*.py',
    '/content/ComfyUI/*.py',
    f'{BASE_BRIDGE}/scripts/*.py',
]
print('🔐 Secreto cargado. Política activa:', ALLOWED_EXECUTABLES, ALLOWED_SCRIPTS)

In [ ]:
# PASO 4: Daemon seguro (verifica firma -> reclama -> ejecuta sin shell)
JOB_ID_RE = re.compile(r'^[A-Za-z0-9_-]{1,64}$')
NONCE_RE = re.compile(r'^[0-9a-f]{16,64}$')
DIRS = {k: f'{BASE_BRIDGE}/{k}' for k in SUBDIRS}
NONCES_FILE = f"{DIRS['state']}/nonces.json"


def glob_match(pattern, path):
    rx = re.escape(pattern).replace(r'\*\*', '.*').replace(r'\*', '[^/]*')
    return re.fullmatch(rx, path) is not None


def canonical(job):
    fields = {k: job.get(k) for k in ('v', 'id', 'action', 'command', 'timestamp', 'nonce')}
    return json.dumps(fields, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')


def write_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)


def load_nonces():
    try:
        with open(NONCES_FILE, 'r', encoding='utf-8') as fh:
            data = json.load(fh)
        now = time.time()
        return {n: exp for n, exp in data.items() if exp > now}
    except Exception:
        return {}


def remember_nonce(nonces, nonce):
    nonces[nonce] = time.time() + 2 * MAX_AGE_S
    write_json(NONCES_FILE, nonces)


def check_command(command):
    if not isinstance(command, str) or not command.strip() or len(command) > 2000 or re.search(r'[\x00-\x1f\x7f]', command):
        return None, 'comando vacío, demasiado largo o con caracteres de control'
    try:
        argv = shlex.split(command)
    except ValueError as exc:
        return None, f'comillas sin cerrar: {exc}'
    if len(argv) < 2 or len(argv) > 60:
        return None, 'formato: python3 /content/script.py [args]'
    if argv[0] not in ALLOWED_EXECUTABLES:
        return None, f'ejecutable no permitido: {argv[0]}'
    script = argv[1]
    if script.startswith('-') or '..' in script.split('/') or not script.endswith('.py'):
        return None, 'script no válido'
    if not any(glob_match(p, script) for p in ALLOWED_SCRIPTS):
        return None, f'script fuera de la allowlist: {script}'
    return argv, None


def verify_job(job, stem, nonces):
    if not isinstance(job, dict) or job.get('v') != 2:
        return None, 'formato no soportado (se exige v=2 firmado)'
    job_id = job.get('id')
    if not isinstance(job_id, str) or not JOB_ID_RE.match(job_id) or job_id != stem:
        return None, 'id no válido o distinto del nombre del archivo'
    if job.get('action') != 'run_command':
        return None, 'acción no permitida'
    ts = job.get('timestamp')
    if not isinstance(ts, int) or isinstance(ts, bool):
        return None, 'timestamp no válido'
    now = time.time()
    if ts > now + 60 or now - ts > MAX_AGE_S:
        return None, 'trabajo caducado (o reloj desincronizado)'
    nonce = job.get('nonce')
    if not isinstance(nonce, str) or not NONCE_RE.match(nonce):
        return None, 'nonce no válido'
    expected = hmac.new(SECRET.encode('utf-8'), canonical(job), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected, str(job.get('sig', ''))):
        return None, 'FIRMA INVÁLIDA: el trabajo no viene de tu PC o fue alterado'
    if nonce in nonces:
        return None, 'nonce repetido (replay)'
    return check_command(job.get('command'))


def reject(path, stem, reason, folder='rejected'):
    safe = re.sub(r'[^A-Za-z0-9_-]', '_', stem)[:64] or 'job'
    print(f'⛔ [{safe}] rechazado: {reason}')
    write_json(f"{DIRS['results']}/{safe}.json", {
        'id': safe, 'status': 'REJECTED', 'reason': reason,
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    shutil.move(path, f"{DIRS[folder]}/{safe}_{int(time.time())}.json")


def run_job(job_id, argv):
    log_path = f"{DIRS['logs']}/{job_id}_cmd.log"
    res_path = f"{DIRS['results']}/{job_id}.json"
    write_json(res_path, {'id': job_id, 'status': 'RUNNING', 'started_at': time.strftime('%Y-%m-%d %H:%M:%S')})
    print(f"\n⚡ Ejecutando [{job_id}]: {shlex.join(argv)[:120]}")
    timed_out = threading.Event()
    proc = subprocess.Popen(
        argv, shell=False, cwd=DIRS['outputs'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    def _kill():
        timed_out.set()
        proc.kill()

    timer = threading.Timer(MAX_JOB_SECONDS, _kill)
    timer.start()
    try:
        with open(log_path, 'w', encoding='utf-8') as log_file:
            for line in proc.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
        proc.wait()
    finally:
        timer.cancel()
    status = 'DONE' if proc.returncode == 0 else ('TIMEOUT' if timed_out.is_set() else 'ERROR')
    write_json(res_path, {
        'id': job_id, 'status': status, 'returncode': proc.returncode,
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    print(f'✅ [{job_id}] terminado: {status}')


print('🎧 Puente seguro activo. Solo se ejecutan trabajos firmados por tu PC...')
nonces = load_nonces()

while True:
    for job_path in sorted(glob.glob(f"{DIRS['jobs']}/*.json")):
        stem = os.path.basename(job_path)[:-5]
        try:
            try:
                with open(job_path, 'r', encoding='utf-8') as fh:
                    job = json.load(fh)
            except Exception as exc:
                if time.time() - os.path.getmtime(job_path) < 30:
                    continue  # Drive puede estar sincronizando todavía
                reject(job_path, stem, f'JSON ilegible: {exc}', 'failed')
                continue

            argv, err = verify_job(job, stem, nonces)
            if err:
                reject(job_path, stem, err)
                continue

            remember_nonce(nonces, job['nonce'])
            # Reclamar ANTES de ejecutar: un trabajo nunca se ejecuta dos veces.
            shutil.move(job_path, f"{DIRS['processed']}/{stem}_{int(time.time())}.json")
            try:
                run_job(stem, argv)
            except Exception as exc:
                write_json(f"{DIRS['results']}/{stem}.json", {
                    'id': stem, 'status': 'ERROR', 'reason': str(exc)[:300],
                    'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
                })
                print(f'❌ [{stem}] error: {exc}')
        except Exception as exc:
            print(f'❌ Error procesando {job_path}: {exc}')
            time.sleep(2)
    time.sleep(2)